# Cross-Modal Contradiction Detection

**Agentic reasoning: how the CAP CDSS catches discrepancies between CXR, labs, and clinical exam.**

> This notebook is generated by `_generate_contradiction_detection_notebook.py`. Do not edit manually.

### 11 Contradiction Rules (CR-1 to CR-11)

The pipeline implements 11 contradiction detection rules that identify discrepancies across
three clinical modalities: chest X-ray, laboratory values, and clinical examination findings.
Rules CR-1 through CR-6 are **cross-modal** (resolved by MedGemma via Strategies A-D),
while CR-7 through CR-11 are **stewardship** rules (resolved deterministically via Strategy E).

### 5 Resolution Strategies

| Strategy | Type | GPU | Description |
|----------|------|-----|-------------|
| A | Zone-specific re-analysis | Yes | MedGemma re-examines CXR region matching clinical signs |
| B | Temporal context assessment | Yes | MedGemma considers early/atypical pneumonia |
| C | Differential diagnosis | Yes | MedGemma evaluates alternative explanations |
| D | Severity override reasoning | Yes | MedGemma assesses hidden severity markers |
| E | Deterministic stewardship | No | Pure Python rule engine (0 GPU calls) |

### Quick Start
1. Set runtime to **GPU -> A100** (Runtime -> Change runtime type)
2. Add **HF_TOKEN** to Colab Secrets (key icon in left sidebar)
3. **Run All** (Runtime -> Run all)


> **Disclaimer:** This notebook demonstrates a research prototype. All clinical outputs
> (severity scores, antibiotic recommendations, contradiction alerts, CXR analysis)
> are **AI-generated by MedGemma 1.5 4B** and have been validated only on synthetic data.
> This system is **not a medical device**, is not approved for clinical use, and must not
> be used for patient care decisions. See [DISCLAIMER.md](../DISCLAIMER.md) for full terms.


In [ ]:
# Cell 1: Install package + dependencies
import os

# Detect Colab vs local
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install from GitHub (requires GITHUB_TOKEN in Colab Secrets)
    try:
        github_token = userdata.get("GITHUB_TOKEN")
        repo_url = f"git+https://{github_token}@github.com/HP-00/MedGemma-Pneumonia-Agent.git@main"
    except Exception:
        repo_url = "git+https://github.com/HP-00/MedGemma-Pneumonia-Agent.git@main"

    %pip install --quiet {repo_url}
    %pip install --quiet plotly>=5.0.0 nest-asyncio
else:
    print("Local environment detected. Ensure: pip install -e '.[dev,benchmark]'")

import nest_asyncio
nest_asyncio.apply()

print("Setup complete")


## Authentication


In [ ]:
import os

# HuggingFace token for gated model access
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    pass

assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set! Add it to Colab Secrets or environment."
print("Authentication complete")


## Load Model & Build Graph


In [ ]:
import time
from cap_agent.models.medgemma import get_model_and_processor
from cap_agent.agent.graph import build_cap_agent_graph

# Load MedGemma (lazy singleton - only loads once, includes warm-up)
model, processor = get_model_and_processor()
print(f"Model loaded on {model.device}")

# Build the 8-node LangGraph pipeline
graph = build_cap_agent_graph()
print("Graph compiled: 8 nodes, conditional routing at contradiction check")


## Contradiction Rules Overview

The 11 contradiction rules detect specific cross-modal discrepancy patterns:

| Rule | Pattern | Confidence | Strategy |
|------|---------|------------|----------|
| CR-1 | CXR negative + focal respiratory signs | high/moderate | A |
| CR-2 | CXR negative + high CRP >100 | high/moderate | B |
| CR-3 | CXR consolidation + low CRP <20 | high | C |
| CR-4 | Low severity + override triggers | high | D |
| CR-5 | Bilateral CXR + unilateral exam | moderate | A |
| CR-6 | Pleural effusion without consolidation | moderate | C |
| CR-7 | Organism-antibiotic mismatch | high | E |
| CR-8 | Unnecessary macrolide | moderate | E |
| CR-9 | IV-to-oral switch opportunity | moderate | E |
| CR-10 | Fluoroquinolone safety concern | high | E |
| CR-11 | Pneumococcal de-escalation | moderate | E |

**Cross-modal rules (CR-1 to CR-6)** compare findings across CXR, labs, and clinical exam.
When fired, MedGemma resolves the discrepancy using Strategies A-D (1 GPU call each).

**Stewardship rules (CR-7 to CR-11)** are deterministic checks on antibiotic prescribing.
Strategy E resolves these with pure Python logic (0 GPU calls).


## Load Group B Cases (Cross-Modal Contradictions)

Group B benchmark cases are specifically designed to trigger cross-modal contradiction
rules CR-1 through CR-6. Each case has known ground truth for expected contradictions.


In [ ]:
from benchmark_data.cases.registry import get_track2_cases

all_track2 = get_track2_cases()

# Group B: cross-modal contradictions (case_id starts with CR1-, CR2-, CR3-, CR4-, CR5, CR6)
group_b = [c for c in all_track2 if any(c["case_id"].startswith(prefix) for prefix in ["CR1-", "CR2-", "CR3-", "CR4-", "CR5", "CR6"])]

print(f"Loaded {len(group_b)} cross-modal contradiction cases")
for c in group_b:
    print(f"  {c['case_id']}: expected contradictions: {c['ground_truth'].get('contradictions', [])}")


In [ ]:
# Clinical output renderer — formats structured_output as a readable document
def render_clinical_output(result, scenario_title="Pipeline Output"):
    """Render structured_output as a formatted clinical document using ANSI codes."""
    ESC = chr(27)
    B = f"{ESC}[1m"       # Bold
    R = f"{ESC}[91m"      # Red
    Y = f"{ESC}[93m"      # Yellow
    G = f"{ESC}[92m"      # Green
    C = f"{ESC}[96m"      # Cyan
    X = f"{ESC}[0m"       # Reset

    so = result.get("structured_output", {})
    sections = so.get("sections", {})

    print(f"\n{B}{C}{'=' * 70}")
    print(f"  CLINICAL DECISION SUPPORT — {scenario_title}")
    print(f"{'=' * 70}{X}")
    print(f"{Y}AI-generated observations for clinician review — not a substitute for clinical judgement.{X}\n")

    # 1. PATIENT
    s1 = sections.get("1_patient", {})
    demo = s1.get("demographics", {})
    print(f"{B}{C}1. PATIENT{X}")
    print(f"  {demo.get('age', '?')}yo {demo.get('sex', '?')}")
    allergy_list = demo.get("allergies", [])
    if allergy_list:
        for a in allergy_list:
            if isinstance(a, dict):
                print(f"  {R}Allergy: {a.get('drug', '?')} — {a.get('reaction_type', '?')} ({a.get('severity', '?')}){X}")
            else:
                print(f"  {R}Allergy: {a}{X}")
    else:
        print(f"  {G}No known drug allergies{X}")
    combos = demo.get("comorbidities", [])
    if combos:
        print(f"  Comorbidities: {', '.join(combos)}")
    if demo.get("pregnancy"):
        print(f"  {R}PREGNANT{X}")
    print(f"  Oral tolerance: {'Yes' if demo.get('oral_tolerance', True) else R + 'No' + X}")
    travel = demo.get("travel_history", [])
    if travel:
        print(f"  {Y}Travel: {', '.join(travel)}{X}")
    print()

    # 2. SEVERITY
    s2 = sections.get("2_severity", {})
    curb = s2.get("curb65", {})
    sev = curb.get("severity_tier", "?")
    sev_color = R if sev == "high" else (Y if sev == "moderate" else G)
    score = curb.get("curb65")
    score_str = f"CURB65={score}" if score is not None else f"CRB65={curb.get('crb65', '?')} (urea unavailable)"
    print(f"{B}{C}2. SEVERITY{X}")
    print(f"  {score_str}  {sev_color}{B}{sev.upper()}{X}")
    print(f"  C={curb.get('c','?')} U={curb.get('u','?')} R={curb.get('r','?')} B={curb.get('b','?')} 65={curb.get('age_65','?')}")
    missing = curb.get("missing_variables", [])
    if missing:
        print(f"  {Y}Missing: {', '.join(missing)}{X}")
    poc = s2.get("place_of_care", {})
    if poc:
        print(f"  Place of care: {poc.get('recommendation', '?')}")
    print()

    # 3. CXR
    s3 = sections.get("3_cxr", {})
    cxr = s3.get("findings", {})
    print(f"{B}{C}3. CHEST X-RAY{X}")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        if not isinstance(f, dict):
            continue
        present = f.get("present", False)
        conf = f.get("confidence", "?")
        if present:
            loc = f.get("location", "")
            bbox = f.get("bounding_box")
            line = f"  {R}PRESENT{X} {finding} ({conf} confidence)"
            if loc:
                line += f" at {loc}"
            if bbox:
                line += f" [bbox: {bbox}]"
            print(line)
        else:
            print(f"  {G}absent{X}  {finding} ({conf} confidence)")
    iq = cxr.get("image_quality", {})
    if iq:
        print(f"  Quality: {iq.get('projection','?')} / {iq.get('rotation','?')} / {iq.get('penetration','?')}")
    longit = cxr.get("longitudinal_comparison")
    if longit:
        print(f"  {B}Longitudinal:{X}")
        for fn, ch in longit.items():
            if isinstance(ch, dict):
                print(f"    {fn}: {ch.get('change','?')} — {ch.get('description','')}")
    print()

    # 4. KEY BLOODS
    s4 = sections.get("4_key_bloods", {})
    labs = s4.get("values", {})
    print(f"{B}{C}4. KEY BLOODS{X}")
    for test, data in (labs or {}).items():
        if not isinstance(data, dict):
            continue
        val = data.get("value", "?")
        unit = data.get("unit", "")
        ref = data.get("reference_range", "")
        abn = data.get("abnormal_flag", False)
        color = R if abn else G
        flag = " ABNORMAL" if abn else ""
        print(f"  {color}{test}: {val} {unit}{flag}{X}  (ref: {ref})")
    print()

    # 5. CONTRADICTIONS
    s5 = sections.get("5_contradiction_alert", {})
    alerts = s5.get("alerts", [])
    informational = s5.get("informational", [])
    resolutions = s5.get("resolutions", [])
    n_total = s5.get("detected", 0)
    print(f"{B}{C}5. CONTRADICTION ALERTS{X}")
    if n_total == 0:
        print(f"  {G}No contradictions detected — findings concordant{X}")
    else:
        print(f"  {n_total} contradiction(s) detected")
        for c in alerts:
            conf = c.get("confidence", "?")
            badge_color = R if conf == "high" else Y
            print(f"  {badge_color}[{conf.upper()}]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
            if c.get("evidence_for"):
                print(f"    For: {c['evidence_for']}")
            if c.get("evidence_against"):
                print(f"    Against: {c['evidence_against']}")
            rec = c.get("recommendation")
            if rec:
                print(f"    {B}Recommendation:{X} {rec}")
        for c in informational:
            print(f"  {G}[LOW]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
        if resolutions:
            print(f"  {B}Resolutions:{X}")
            for r in resolutions:
                print(f"    {r[:200]}")
    print()

    # 6. TREATMENT
    s6 = sections.get("6_treatment_pathway", {})
    abx = s6.get("antibiotic", {})
    print(f"{B}{C}6. TREATMENT{X}")
    print(f"  First-line: {abx.get('first_line', 'N/A')}")
    print(f"  Dose/route: {abx.get('dose_route', 'N/A')}")
    if abx.get("allergy_adjustment"):
        print(f"  {Y}Allergy adj: {abx['allergy_adjustment']}{X}")
    if abx.get("atypical_cover"):
        print(f"  Atypical: {abx['atypical_cover']}")
    if abx.get("renal_adjustment"):
        print(f"  {Y}Renal: {abx['renal_adjustment']}{X}")
    cortico = s6.get("corticosteroid")
    if cortico:
        print(f"  Corticosteroid: {cortico}")
    steward = abx.get("stewardship_notes", [])
    for note in steward:
        print(f"  {Y}Stewardship: {note}{X}")
    inv = s6.get("investigations", {})
    if inv:
        print(f"  {B}Investigations:{X}")
        for name, detail in inv.items():
            if isinstance(detail, dict):
                rec = "Recommended" if detail.get("recommended") else "Not indicated"
                print(f"    {name}: {rec} — {detail.get('reasoning', '')[:100]}")
    print()

    # 7. DATA GAPS
    s7 = sections.get("7_data_gaps", {})
    gaps = s7.get("gaps", [])
    print(f"{B}{C}7. DATA GAPS{X}")
    if gaps:
        for g in gaps:
            print(f"  {Y}- {g}{X}")
    else:
        print(f"  {G}None identified{X}")
    print()

    # 8. MONITORING
    s8 = sections.get("8_monitoring", {})
    plan = s8.get("plan", {})
    print(f"{B}{C}8. MONITORING{X}")
    crp_trend = plan.get("crp_trend")
    if crp_trend:
        adm = crp_trend.get("admission_value", "?")
        cur = crp_trend.get("current_value", "?")
        pct = crp_trend.get("percent_change", "?")
        trend = crp_trend.get("trend", "?")
        sr = crp_trend.get("flag_senior_review", False)
        sr_color = R if sr else G
        print(f"  CRP: {adm} -> {cur} mg/L ({pct}% change, {trend})")
        print(f"  Senior review: {sr_color}{sr}{X}")
    tr = plan.get("treatment_response")
    if tr:
        reassess = tr.get("reassess_needed", False)
        ra_color = R if reassess else G
        print(f"  Treatment response: {ra_color}reassess_needed={reassess}{X}")
        for action in tr.get("actions", []):
            print(f"    - {action}")
    dc = plan.get("discharge_criteria_met")
    if dc is not None:
        dc_color = G if dc else R
        dc_str = "MET" if dc else "NOT MET"
        print(f"  Discharge: {dc_color}{dc_str}{X}")
    dc_detail = plan.get("discharge_criteria_details", {})
    if dc_detail:
        for check, passed_val in dc_detail.items():
            if check.startswith("_"):
                continue
            chk_color = G if passed_val else R
            chk_str = "PASS" if passed_val else "FAIL"
            print(f"    {chk_color}[{chk_str}]{X} {check}")
    print(f"  Next review: {plan.get('next_review', 'N/A')}")
    print()

    # PROVENANCE
    prov = so.get("provenance", {})
    if prov:
        print(f"{B}{C}PROVENANCE{X}")
        tools = prov.get("extraction_tools", {})
        sources = prov.get("data_sources", {})
        for tool_name, pipeline in tools.items():
            src = sources.get(tool_name, [])
            print(f"  {tool_name}: {pipeline} ({', '.join(src) if src else 'N/A'})")

    print(f"\n{C}{'=' * 70}{X}\n")

print("render_clinical_output() defined")


## Run All Group B Cases

Execute the full 8-node pipeline for each cross-modal contradiction case.
Each case uses mock extraction (no CXR images, no FHIR) so the contradiction
detection logic is tested against known synthetic inputs.


In [ ]:
from cap_agent.agent.state import build_initial_state
import time

all_results = []
for i, case in enumerate(group_b):
    case_id = case["case_id"]
    print(f"[{i+1}/{len(group_b)}] Running {case_id}...")
    initial_state = build_initial_state(case)
    start = time.time()
    result = None
    for chunk in graph.stream(initial_state, stream_mode="values"):
        result = chunk
    elapsed = time.time() - start
    contradictions = result.get("contradictions_detected", [])
    cr_ids = [c["rule_id"] for c in contradictions]
    all_results.append({"case_id": case_id, "case": case, "result": result, "elapsed": elapsed})
    print(f"  Detected: {cr_ids} in {elapsed:.1f}s")

print(f"\nAll {len(all_results)} cases complete")


## CR-1: CXR Negative + Focal Respiratory Signs

**Pattern:** Chest X-ray shows no consolidation, but clinical examination reveals focal
crackles and/or bronchial breathing — suggesting pneumonia the CXR may have missed.

**Confidence:**
- **High** when both crackles AND bronchial breathing are present
- **Moderate** when only one sign is present

**Resolution:** Strategy A — MedGemma re-examines the CXR region matching the clinical signs,
looking for subtle opacification or early infiltrate that may not meet the consolidation threshold.


In [ ]:
rule_cases = [r for r in all_results if r["case_id"].startswith("CR1-")]
for r in rule_cases:
    result = r["result"]
    contradictions = result.get("contradictions_detected", [])
    resolutions = result.get("resolution_results", [])

    print(f"\n{'='*60}")
    print(f"Case: {r['case_id']} ({r['elapsed']:.1f}s)")
    print(f"{'='*60}")

    print(f"\nDetected contradictions:")
    for c in contradictions:
        print(f"  [{c.get('confidence', '?').upper()}] {c['rule_id']}: {c['pattern']}")
        if c.get("recommendation"):
            print(f"    Recommendation: {c['recommendation']}")

    if resolutions:
        print(f"\nMedGemma Resolution:")
        for res in resolutions:
            print(f"  {res[:300]}")

    # Show thinking trace if available
    thinking = result.get("thinking_traces", [])
    if thinking:
        print(f"\nThinking trace ({len(thinking)} entries):")
        for t in thinking[-2:]:  # Last 2 entries
            print(f"  {t[:200]}...")

    render_clinical_output(result, r["case_id"])


## CR-2: CXR Negative + High CRP >100

**Pattern:** Chest X-ray shows no consolidation, but CRP is significantly elevated (>100 mg/L),
suggesting an inflammatory process the CXR has not yet captured (early pneumonia, atypical pathogen).

**Confidence:**
- **High** when CRP > 200
- **Moderate** when CRP 100-200

**Resolution:** Strategy B — MedGemma performs temporal context assessment, considering whether
this could represent early pneumonia (CXR lags behind clinical/biochemical changes by 12-24h)
or an atypical pathogen with different imaging characteristics.

**Note:** CR-1 also fires on these cases because the no-consolidation + clinical signs
prerequisite is shared.


In [ ]:
rule_cases = [r for r in all_results if r["case_id"].startswith("CR2-")]
for r in rule_cases:
    result = r["result"]
    contradictions = result.get("contradictions_detected", [])
    resolutions = result.get("resolution_results", [])

    print(f"\n{'='*60}")
    print(f"Case: {r['case_id']} ({r['elapsed']:.1f}s)")
    print(f"{'='*60}")

    print(f"\nDetected contradictions:")
    for c in contradictions:
        print(f"  [{c.get('confidence', '?').upper()}] {c['rule_id']}: {c['pattern']}")
        if c.get("recommendation"):
            print(f"    Recommendation: {c['recommendation']}")

    if resolutions:
        print(f"\nMedGemma Resolution:")
        for res in resolutions:
            print(f"  {res[:300]}")

    # Show thinking trace if available
    thinking = result.get("thinking_traces", [])
    if thinking:
        print(f"\nThinking trace ({len(thinking)} entries):")
        for t in thinking[-2:]:  # Last 2 entries
            print(f"  {t[:200]}...")

    render_clinical_output(result, r["case_id"])


## CR-3: CXR Consolidation + Low CRP <20

**Pattern:** Chest X-ray shows consolidation, but CRP is surprisingly low (<20 mg/L),
raising the question of whether the CXR finding represents true active infection
or an alternative diagnosis (old scarring, atelectasis, malignancy).

**Confidence:** High (CRP <10 is a strong discrepancy signal)

**Resolution:** Strategy C — MedGemma performs differential diagnosis, evaluating
whether the consolidation could represent a non-infectious process and recommending
further investigations (CT, bronchoscopy) if appropriate.


In [ ]:
rule_cases = [r for r in all_results if r["case_id"].startswith("CR3-")]
for r in rule_cases:
    result = r["result"]
    contradictions = result.get("contradictions_detected", [])
    resolutions = result.get("resolution_results", [])

    print(f"\n{'='*60}")
    print(f"Case: {r['case_id']} ({r['elapsed']:.1f}s)")
    print(f"{'='*60}")

    print(f"\nDetected contradictions:")
    for c in contradictions:
        print(f"  [{c.get('confidence', '?').upper()}] {c['rule_id']}: {c['pattern']}")
        if c.get("recommendation"):
            print(f"    Recommendation: {c['recommendation']}")

    if resolutions:
        print(f"\nMedGemma Resolution:")
        for res in resolutions:
            print(f"  {res[:300]}")

    # Show thinking trace if available
    thinking = result.get("thinking_traces", [])
    if thinking:
        print(f"\nThinking trace ({len(thinking)} entries):")
        for t in thinking[-2:]:  # Last 2 entries
            print(f"  {t[:200]}...")

    render_clinical_output(result, r["case_id"])


## CR-4: Low Severity + Override Triggers

**Pattern:** CURB65 scores low severity, but the patient has features suggesting
they may be sicker than the score indicates: immunosuppression, bilateral consolidation,
multilobar disease, or sepsis markers.

**Confidence:** High (safety-critical — under-triaging a sick patient)

**Resolution:** Strategy D — MedGemma performs severity override reasoning, assessing
whether the CURB65 score should be overridden by these additional risk factors and
recommending escalation to hospital-level care if appropriate.


In [ ]:
rule_cases = [r for r in all_results if r["case_id"].startswith("CR4-")]
for r in rule_cases:
    result = r["result"]
    contradictions = result.get("contradictions_detected", [])
    resolutions = result.get("resolution_results", [])

    print(f"\n{'='*60}")
    print(f"Case: {r['case_id']} ({r['elapsed']:.1f}s)")
    print(f"{'='*60}")

    print(f"\nDetected contradictions:")
    for c in contradictions:
        print(f"  [{c.get('confidence', '?').upper()}] {c['rule_id']}: {c['pattern']}")
        if c.get("recommendation"):
            print(f"    Recommendation: {c['recommendation']}")

    if resolutions:
        print(f"\nMedGemma Resolution:")
        for res in resolutions:
            print(f"  {res[:300]}")

    # Show thinking trace if available
    thinking = result.get("thinking_traces", [])
    if thinking:
        print(f"\nThinking trace ({len(thinking)} entries):")
        for t in thinking[-2:]:  # Last 2 entries
            print(f"  {t[:200]}...")

    render_clinical_output(result, r["case_id"])


## CR-5: Bilateral CXR + Unilateral Exam

**Pattern:** Chest X-ray shows bilateral consolidation, but clinical examination
only reveals signs on one side — suggesting either the CXR is over-reporting
or the exam missed contralateral signs.

**Confidence:** Moderate

**Resolution:** Strategy A — MedGemma re-examines the CXR, focusing on the side
without clinical signs to assess whether the bilateral finding is genuine or an artifact.


In [ ]:
rule_cases = [r for r in all_results if r["case_id"].startswith("CR5")]
for r in rule_cases:
    result = r["result"]
    contradictions = result.get("contradictions_detected", [])
    resolutions = result.get("resolution_results", [])

    print(f"\n{'='*60}")
    print(f"Case: {r['case_id']} ({r['elapsed']:.1f}s)")
    print(f"{'='*60}")

    print(f"\nDetected contradictions:")
    for c in contradictions:
        print(f"  [{c.get('confidence', '?').upper()}] {c['rule_id']}: {c['pattern']}")
        if c.get("recommendation"):
            print(f"    Recommendation: {c['recommendation']}")

    if resolutions:
        print(f"\nMedGemma Resolution:")
        for res in resolutions:
            print(f"  {res[:300]}")

    # Show thinking trace if available
    thinking = result.get("thinking_traces", [])
    if thinking:
        print(f"\nThinking trace ({len(thinking)} entries):")
        for t in thinking[-2:]:  # Last 2 entries
            print(f"  {t[:200]}...")

    render_clinical_output(result, r["case_id"])


## CR-6: Pleural Effusion Without Consolidation

**Pattern:** Chest X-ray shows pleural effusion but no consolidation — raising
the question of whether this represents a parapneumonic effusion (with pneumonia
hidden behind the fluid) or a non-infectious cause (heart failure, malignancy).

**Confidence:** Moderate

**Resolution:** Strategy C — MedGemma performs differential diagnosis, considering
the clinical context to determine the most likely aetiology of the effusion
and recommending further investigations (pleural aspiration, CT, echocardiography).


In [ ]:
rule_cases = [r for r in all_results if r["case_id"].startswith("CR6")]
for r in rule_cases:
    result = r["result"]
    contradictions = result.get("contradictions_detected", [])
    resolutions = result.get("resolution_results", [])

    print(f"\n{'='*60}")
    print(f"Case: {r['case_id']} ({r['elapsed']:.1f}s)")
    print(f"{'='*60}")

    print(f"\nDetected contradictions:")
    for c in contradictions:
        print(f"  [{c.get('confidence', '?').upper()}] {c['rule_id']}: {c['pattern']}")
        if c.get("recommendation"):
            print(f"    Recommendation: {c['recommendation']}")

    if resolutions:
        print(f"\nMedGemma Resolution:")
        for res in resolutions:
            print(f"  {res[:300]}")

    # Show thinking trace if available
    thinking = result.get("thinking_traces", [])
    if thinking:
        print(f"\nThinking trace ({len(thinking)} entries):")
        for t in thinking[-2:]:  # Last 2 entries
            print(f"  {t[:200]}...")

    render_clinical_output(result, r["case_id"])


## Resolution Strategy Summary

The pipeline uses two fundamentally different resolution approaches:

### GPU-Based Resolution (Strategies A-D)
Cross-modal contradictions (CR-1 to CR-6) are resolved by MedGemma, which reasons
about the discrepancy using the full clinical context. Each resolution costs 1 GPU call.

| Strategy | Applied To | What MedGemma Does |
|----------|-----------|-------------------|
| A | CR-1, CR-5 | Zone-specific CXR re-analysis matching clinical signs |
| B | CR-2 | Temporal context: early pneumonia, CXR lag behind clinical/lab changes |
| C | CR-3, CR-6 | Differential diagnosis: alternative explanations for discrepant findings |
| D | CR-4 | Severity override: hidden risk factors the score misses |

### Deterministic Resolution (Strategy E)
Stewardship contradictions (CR-7 to CR-11) are resolved by pure Python logic
with zero GPU calls. These rules check antibiotic prescribing against microbiology
results and clinical guidelines.

| Rule | What Strategy E Checks |
|------|----------------------|
| CR-7 | Organism susceptibility vs prescribed antibiotic |
| CR-8 | Macrolide necessity given microbiology results |
| CR-9 | IV-to-oral switch eligibility (>48h IV + clinically stable) |
| CR-10 | Fluoroquinolone prescribing with penicillin intolerance (not true allergy) |
| CR-11 | Pneumococcal de-escalation opportunity |


## Contradiction Recall & Precision

Per-rule recall and precision across all Group B cases. Recall measures whether
expected contradictions were detected; precision measures whether detected
contradictions were actually expected.


In [ ]:
import plotly.graph_objects as go

# Collect all rule IDs
all_expected = set()
all_actual = set()
for r in all_results:
    gt = r["case"].get("ground_truth", {})
    all_expected |= set(gt.get("contradictions", []))
    detected = r["result"].get("contradictions_detected", [])
    all_actual |= {c["rule_id"] for c in detected}

all_rules = sorted(all_expected | all_actual)

recalls = []
precisions = []
for rule in all_rules:
    tp = 0
    fn = 0
    fp = 0
    for r in all_results:
        gt = r["case"].get("ground_truth", {})
        expected_rules = set(gt.get("contradictions", []))
        detected_rules = {c["rule_id"] for c in r["result"].get("contradictions_detected", [])}

        if rule in expected_rules and rule in detected_rules:
            tp += 1
        elif rule in expected_rules and rule not in detected_rules:
            fn += 1
        elif rule not in expected_rules and rule in detected_rules:
            fp += 1

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recalls.append(recall)
    precisions.append(precision)
    print(f"  {rule}: recall={recall:.0%} precision={precision:.0%} (TP={tp} FN={fn} FP={fp})")

# Plotly grouped bar chart
fig = go.Figure(data=[
    go.Bar(name="Recall", x=all_rules, y=recalls, marker_color="#2196F3"),
    go.Bar(name="Precision", x=all_rules, y=precisions, marker_color="#FF9800"),
])
fig.update_layout(
    title="Contradiction Detection: Recall & Precision per Rule",
    barmode="group",
    yaxis=dict(range=[0, 1.05], title="Score"),
    xaxis_title="Contradiction Rule",
    width=700, height=400,
)
fig.show()

# Overall summary
total_tp = sum(1 for r in all_results
               for rule in r["case"].get("ground_truth", {}).get("contradictions", [])
               if rule in {c["rule_id"] for c in r["result"].get("contradictions_detected", [])})
total_expected = sum(len(r["case"].get("ground_truth", {}).get("contradictions", []))
                     for r in all_results)
total_detected = sum(len(r["result"].get("contradictions_detected", []))
                     for r in all_results)
overall_recall = total_tp / total_expected if total_expected > 0 else 0
overall_precision = total_tp / total_detected if total_detected > 0 else 0

print(f"\nOverall: recall={overall_recall:.0%} precision={overall_precision:.0%}")
print(f"  {total_tp}/{total_expected} expected contradictions detected")
print(f"  {total_tp}/{total_detected} detected contradictions were expected")


## Verification Checklist

Expected vs detected contradictions per case. Each case must detect all expected
contradiction rules to pass.


In [ ]:
ESC = chr(27)
B = f"{ESC}[1m"; G = f"{ESC}[92m"; R = f"{ESC}[91m"; Y = f"{ESC}[93m"
C = f"{ESC}[96m"; X = f"{ESC}[0m"

print(f"\n{B}{C}{'=' * 70}")
print(f"  CONTRADICTION DETECTION VERIFICATION")
print(f"{'=' * 70}{X}\n")

total_pass = 0
total_fail = 0

for r in all_results:
    case_id = r["case_id"]
    gt = r["case"].get("ground_truth", {})
    expected = set(gt.get("contradictions", []))
    detected = {c["rule_id"] for c in r["result"].get("contradictions_detected", [])}

    missing = expected - detected
    unexpected = detected - expected
    all_found = len(missing) == 0

    status_color = G if all_found else R
    status = "PASS" if all_found else "FAIL"

    if all_found:
        total_pass += 1
    else:
        total_fail += 1

    print(f"  {status_color}[{status}]{X} {B}{case_id}{X}")
    print(f"         Expected:   {sorted(expected)}")
    print(f"         Detected:   {sorted(detected)}")
    if missing:
        print(f"         {R}Missing:    {sorted(missing)}{X}")
    if unexpected:
        print(f"         {Y}Unexpected: {sorted(unexpected)}{X}")
    print()

print(f"{B}Results: {total_pass}/{total_pass + total_fail} cases passed{X}")
total_time = sum(r["elapsed"] for r in all_results)
print(f"Total pipeline time: {total_time:.1f}s ({total_time / len(all_results):.1f}s per case)")

if total_fail == 0:
    print(f"\n{G}{B}All cross-modal contradiction checks passed!{X}")
else:
    print(f"\n{R}{B}{total_fail} case(s) failed — review missing contradictions above.{X}")
